# Quilt-LLaVA batch commands

Run the configuration cell, then copy the output of the desired cell into a shell at the MedVLMBench repository root. These cells **print commands only**. Set dataset paths, experiment paths, cache and checkpoint first. The official checkpoint is used for off-the-shelf (OTS) evaluation; adapted evaluations only include discovered LoRA adapters with an `adapter_config.json`.

Quilt-LLaVA is a LLaVA-1.5 single-image model. MedXpertQA is omitted from the default batch because multi-image questions need a separate compatibility check.

In [27]:
from pathlib import Path
import os
import shlex

# Run the printed commands from the MedVLMBench repository root.
EXP_PATH = os.environ.get("MEDVLMBENCH_EXP_PATH", "/research/d5/gds/yzhong22/experiments/med_vlm_benchmark")
DISCOVERY_ROOT = os.environ.get("MEDVLMBENCH_DISCOVERY_ROOT", "/media/yesindeed/DATADRIVE1/mount/remote_cse/experiments/med_vlm_benchmark")
CACHE_DIR = os.environ.get("HF_HOME", "/research/d5/gds/yzhong22/misc/cache")
MODEL_PATH = os.environ.get("QUILT_LLAVA_MODEL_PATH", "/research/d5/gds/yzhong22/misc/pretrained/Quilt-Llava-v1.5-7b")
SCRIPT = "./script/yuan/train_infer_bash/quilt_llava/quilt_llava_vqa.sh"
MODEL_NAME = "Quilt-LLaVA"
SEED = int(os.environ.get("MEDVLMBENCH_SEED", "42"))
DATASETS = {
    "SLAKE": "/research/d5/gds/yzhong22/datasets/SLAKE/imgs",
    "PathVQA": "None",
    "VQA-RAD": "None",
    "Harvard-FairVLMed10k": "/research/d5/gds/yzhong22/datasets/Harvard-FairVLMed10k",
    "OmniMedVQA": "/research/d5/gds/yzhong22/datasets/OmniMedVQA",
}

def command(mode, dataset, checkpoint=None, model_base=None, extra_env=None):
    if dataset not in DATASETS:
        raise ValueError(f"Unknown dataset: {dataset}")
    variables = {
        "MODE": mode, "DATASET": dataset, "IMAGE_PATH": DATASETS[dataset],
        "MODEL_PATH": checkpoint or MODEL_PATH, "EXP_PATH": EXP_PATH,
        "SEED": str(SEED), "HF_HOME": CACHE_DIR,
    }
    if mode != "train":
        variables["SPLIT"] = "test"
    if model_base is not None:
        variables["MODEL_BASE"] = model_base
    variables.update(extra_env or {})
    return shlex.join(["env", *[f"{key}={value}" for key, value in variables.items()], "bash", SCRIPT])

def emit(commands):
    print(" &&\n".join(commands) if commands else "No matching commands; check the configured datasets/checkpoints.")

def discover_adapted_checkpoints():
    # Only use completed adapter directories. The training output is saved under
    # EXP_PATH/vqa/<dataset>/<model>/train_*; a mirror can be scanned separately.
    found = {}
    for dataset in DATASETS:
        source = Path(DISCOVERY_ROOT) / "vqa" / dataset / MODEL_NAME
        paths = []
        for directory in sorted(source.glob("train_*")):
            if not directory.is_dir() or "backup" in directory.name:
                continue
            if not (directory / "adapter_config.json").is_file():
                continue
            paths.append(str(Path(EXP_PATH) / "vqa" / dataset / MODEL_NAME / directory.name))
        if paths:
            found[dataset] = paths
    return found

# Set explicit adapter paths here when discovery is unavailable, e.g.
# ADAPTED_CHECKPOINTS = {"OmniMedVQA": ["/path/to/adapter"]}
ADAPTED_CHECKPOINTS = discover_adapted_checkpoints()


## Official checkpoint: direct VQA evaluation

In [34]:
DATASETS1 = {
    "SLAKE": "/research/d5/gds/yzhong22/datasets/SLAKE/imgs",
    # "PathVQA": "None",
    "VQA-RAD": "None",
    "Harvard-FairVLMed10k": "/research/d5/gds/yzhong22/datasets/Harvard-FairVLMed10k",
    "OmniMedVQA": "/research/d5/gds/yzhong22/datasets/OmniMedVQA",
}


emit([command("eval", dataset) for dataset in DATASETS1])

env MODE=eval DATASET=SLAKE IMAGE_PATH=/research/d5/gds/yzhong22/datasets/SLAKE/imgs MODEL_PATH=/research/d5/gds/yzhong22/misc/pretrained/Quilt-Llava-v1.5-7b EXP_PATH=/research/d5/gds/yzhong22/experiments/med_vlm_benchmark SEED=42 HF_HOME=/research/d5/gds/yzhong22/misc/cache SPLIT=test bash ./script/yuan/train_infer_bash/quilt_llava/quilt_llava_vqa.sh &&
env MODE=eval DATASET=VQA-RAD IMAGE_PATH=None MODEL_PATH=/research/d5/gds/yzhong22/misc/pretrained/Quilt-Llava-v1.5-7b EXP_PATH=/research/d5/gds/yzhong22/experiments/med_vlm_benchmark SEED=42 HF_HOME=/research/d5/gds/yzhong22/misc/cache SPLIT=test bash ./script/yuan/train_infer_bash/quilt_llava/quilt_llava_vqa.sh &&
env MODE=eval DATASET=Harvard-FairVLMed10k IMAGE_PATH=/research/d5/gds/yzhong22/datasets/Harvard-FairVLMed10k MODEL_PATH=/research/d5/gds/yzhong22/misc/pretrained/Quilt-Llava-v1.5-7b EXP_PATH=/research/d5/gds/yzhong22/experiments/med_vlm_benchmark SEED=42 HF_HOME=/research/d5/gds/yzhong22/misc/cache SPLIT=test bash ./script

## Official checkpoint: MDAgents

In [35]:
DATASETS1 = {
    "SLAKE": "/research/d5/gds/yzhong22/datasets/SLAKE/imgs",
    "PathVQA": "None",
    "VQA-RAD": "None",
    "Harvard-FairVLMed10k": "/research/d5/gds/yzhong22/datasets/Harvard-FairVLMed10k",
    "OmniMedVQA": "/research/d5/gds/yzhong22/datasets/OmniMedVQA",
}

emit([command("mdagent", dataset) for dataset in DATASETS1])

env MODE=mdagent DATASET=SLAKE IMAGE_PATH=/research/d5/gds/yzhong22/datasets/SLAKE/imgs MODEL_PATH=/research/d5/gds/yzhong22/misc/pretrained/Quilt-Llava-v1.5-7b EXP_PATH=/research/d5/gds/yzhong22/experiments/med_vlm_benchmark SEED=42 HF_HOME=/research/d5/gds/yzhong22/misc/cache SPLIT=test bash ./script/yuan/train_infer_bash/quilt_llava/quilt_llava_vqa.sh &&
env MODE=mdagent DATASET=PathVQA IMAGE_PATH=None MODEL_PATH=/research/d5/gds/yzhong22/misc/pretrained/Quilt-Llava-v1.5-7b EXP_PATH=/research/d5/gds/yzhong22/experiments/med_vlm_benchmark SEED=42 HF_HOME=/research/d5/gds/yzhong22/misc/cache SPLIT=test bash ./script/yuan/train_infer_bash/quilt_llava/quilt_llava_vqa.sh &&
env MODE=mdagent DATASET=VQA-RAD IMAGE_PATH=None MODEL_PATH=/research/d5/gds/yzhong22/misc/pretrained/Quilt-Llava-v1.5-7b EXP_PATH=/research/d5/gds/yzhong22/experiments/med_vlm_benchmark SEED=42 HF_HOME=/research/d5/gds/yzhong22/misc/cache SPLIT=test bash ./script/yuan/train_infer_bash/quilt_llava/quilt_llava_vqa.sh &

## Official checkpoint: UCAgents

In [36]:
DATASETS1 = {
    "SLAKE": "/research/d5/gds/yzhong22/datasets/SLAKE/imgs",
    "PathVQA": "None",
    "VQA-RAD": "None",
    "Harvard-FairVLMed10k": "/research/d5/gds/yzhong22/datasets/Harvard-FairVLMed10k",
    "OmniMedVQA": "/research/d5/gds/yzhong22/datasets/OmniMedVQA",
}

emit([command("ucagent", dataset) for dataset in DATASETS1])

env MODE=ucagent DATASET=SLAKE IMAGE_PATH=/research/d5/gds/yzhong22/datasets/SLAKE/imgs MODEL_PATH=/research/d5/gds/yzhong22/misc/pretrained/Quilt-Llava-v1.5-7b EXP_PATH=/research/d5/gds/yzhong22/experiments/med_vlm_benchmark SEED=42 HF_HOME=/research/d5/gds/yzhong22/misc/cache SPLIT=test bash ./script/yuan/train_infer_bash/quilt_llava/quilt_llava_vqa.sh &&
env MODE=ucagent DATASET=PathVQA IMAGE_PATH=None MODEL_PATH=/research/d5/gds/yzhong22/misc/pretrained/Quilt-Llava-v1.5-7b EXP_PATH=/research/d5/gds/yzhong22/experiments/med_vlm_benchmark SEED=42 HF_HOME=/research/d5/gds/yzhong22/misc/cache SPLIT=test bash ./script/yuan/train_infer_bash/quilt_llava/quilt_llava_vqa.sh &&
env MODE=ucagent DATASET=VQA-RAD IMAGE_PATH=None MODEL_PATH=/research/d5/gds/yzhong22/misc/pretrained/Quilt-Llava-v1.5-7b EXP_PATH=/research/d5/gds/yzhong22/experiments/med_vlm_benchmark SEED=42 HF_HOME=/research/d5/gds/yzhong22/misc/cache SPLIT=test bash ./script/yuan/train_infer_bash/quilt_llava/quilt_llava_vqa.sh &

## Adapted LoRA checkpoints: direct VQA evaluation

Only directories containing `adapter_config.json` are included. `MODEL_BASE` points to the official checkpoint. If checkpoints are on another mount, change `DISCOVERY_ROOT`; if needed, assign `ADAPTED_CHECKPOINTS` explicitly in the configuration cell.

In [31]:
emit([command("eval", dataset, checkpoint=checkpoint, model_base=MODEL_PATH)
      for dataset, checkpoints in ADAPTED_CHECKPOINTS.items() for checkpoint in checkpoints])

env MODE=eval DATASET=SLAKE IMAGE_PATH=/research/d5/gds/yzhong22/datasets/SLAKE/imgs MODEL_PATH=/research/d5/gds/yzhong22/experiments/med_vlm_benchmark/vqa/SLAKE/Quilt-LLaVA/train_lora_ML_seed42_quilt_llava EXP_PATH=/research/d5/gds/yzhong22/experiments/med_vlm_benchmark SEED=42 HF_HOME=/research/d5/gds/yzhong22/misc/cache SPLIT=test MODEL_BASE=/research/d5/gds/yzhong22/misc/pretrained/Quilt-Llava-v1.5-7b bash ./script/yuan/train_infer_bash/quilt_llava/quilt_llava_vqa.sh &&
env MODE=eval DATASET=PathVQA IMAGE_PATH=None MODEL_PATH=/research/d5/gds/yzhong22/experiments/med_vlm_benchmark/vqa/PathVQA/Quilt-LLaVA/train_lora_ML_seed42_quilt_llava EXP_PATH=/research/d5/gds/yzhong22/experiments/med_vlm_benchmark SEED=42 HF_HOME=/research/d5/gds/yzhong22/misc/cache SPLIT=test MODEL_BASE=/research/d5/gds/yzhong22/misc/pretrained/Quilt-Llava-v1.5-7b bash ./script/yuan/train_infer_bash/quilt_llava/quilt_llava_vqa.sh &&
env MODE=eval DATASET=VQA-RAD IMAGE_PATH=None MODEL_PATH=/research/d5/gds/yzhon

## Adapted LoRA checkpoints: MDAgents

In [14]:


emit([command("mdagent", dataset, checkpoint=checkpoint, model_base=MODEL_PATH)
      for dataset, checkpoints in ADAPTED_CHECKPOINTS.items() for checkpoint in checkpoints])

No matching commands; check the configured datasets/checkpoints.


## Adapted LoRA checkpoints: UCAgents

In [ ]:
emit([command("ucagent", dataset, checkpoint=checkpoint, model_base=MODEL_PATH)
      for dataset, checkpoints in ADAPTED_CHECKPOINTS.items() for checkpoint in checkpoints])

## Framework LoRA training

Uses the Quilt-LLaVA wrapper and `run_train.py`. Select datasets deliberately, then keep trained and OTS results separate.

In [7]:
# TRAIN_DATASETS = ["OmniMedVQA"]
emit([command("train", dataset) for dataset in DATASETS])

env MODE=train DATASET=SLAKE IMAGE_PATH=/research/d5/gds/yzhong22/datasets/SLAKE/imgs MODEL_PATH=/research/d5/gds/yzhong22/misc/pretrained/Quilt-Llava-v1.5-7b EXP_PATH=/research/d5/gds/yzhong22/experiments/med_vlm_benchmark SEED=42 HF_HOME=/research/d5/gds/yzhong22/misc/cache bash ./script/yuan/train_infer_bash/quilt_llava/quilt_llava_vqa.sh &&
env MODE=train DATASET=PathVQA IMAGE_PATH=None MODEL_PATH=/research/d5/gds/yzhong22/misc/pretrained/Quilt-Llava-v1.5-7b EXP_PATH=/research/d5/gds/yzhong22/experiments/med_vlm_benchmark SEED=42 HF_HOME=/research/d5/gds/yzhong22/misc/cache bash ./script/yuan/train_infer_bash/quilt_llava/quilt_llava_vqa.sh &&
env MODE=train DATASET=VQA-RAD IMAGE_PATH=None MODEL_PATH=/research/d5/gds/yzhong22/misc/pretrained/Quilt-Llava-v1.5-7b EXP_PATH=/research/d5/gds/yzhong22/experiments/med_vlm_benchmark SEED=42 HF_HOME=/research/d5/gds/yzhong22/misc/cache bash ./script/yuan/train_infer_bash/quilt_llava/quilt_llava_vqa.sh &&
env MODE=train DATASET=Harvard-FairVL